# Bước 2 trên Kaggle — Phạm Văn Hoàng Anh Tú (2A202602507)

Colab bản miễn phí hết lượt GPU, nên Bước 2 chạy trên Kaggle (GPU **T4**, giống loại GPU đã dùng trên Colab).
Notebook chạy được từ đầu đến cuối không cần bấm gì (dùng **Save Version → Save & Run All**).

- Code lấy từ GitHub (repo công khai), không cần token.
- Ảnh tải từ Zenodo, kiểm tra MD5, giải nén vào đĩa tạm `/tmp/data` (không nằm trong Output).
- Kết quả nhỏ (log, biểu đồ, logit val, `done.json`) được gói vào `/kaggle/working/kaggle_buoc2.zip`;
  checkpoint `.pt` để lại trong Output, không cho vào zip.
- Cả 3 seed của mốc T00 đều train trên Kaggle để mọi thí nghiệm Bước 2 cùng một môi trường.

In [ ]:
# Ô K1 — kiểm tra GPU, cài thư viện, in phiên bản
import os, sys, platform, subprocess
LOCAL_TEST = os.environ.get("LAB_LOCAL_TEST") == "1"   # chỉ Claude dùng khi chạy thử trên CPU; trên Kaggle luôn False
if not LOCAL_TEST:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "timm", "openpyxl"], check=True)
import numpy as np, pandas as pd, torch, torchvision, timm
print("python", platform.python_version(), "| torch", torch.__version__, "| torchvision", torchvision.__version__,
      "| timm", timm.__version__, "| numpy", np.__version__, "| pandas", pd.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")
assert LOCAL_TEST or torch.cuda.is_available(), "Chưa bật GPU: Settings → Accelerator → GPU T4 x2"

In [ ]:
# Ô K2 — lấy code từ GitHub (nhánh có code đã chạy thử)
ROOT = os.environ.get("LAB_ROOT", "/kaggle")
WORK = f"{ROOT}/working"
REPO = f"{WORK}/repo"
BRANCH = "claude/project-thread-mdpq02"
if not LOCAL_TEST and not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH,
                    "https://github.com/tubepvhat1604-bot/K4-DAY02-PhamVanHoangAnhTu-2A202602507.git", REPO], check=True)
CODE_DIR = f"{REPO}/submissions/2A202602507_pham_van_hoang_anh_tu/code"
for p in (REPO, CODE_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)
OUT = f"{WORK}/sub"                      # giống cấu trúc thư mục bài nộp: curves/, predictions/, logs/
RUNS_DIR = f"{WORK}/deepweeds_runs"      # checkpoint + log từng run
for d in ("curves", "predictions", "logs"):
    os.makedirs(f"{OUT}/{d}", exist_ok=True)
os.chdir(OUT)
import train, dataset
from train import Config, run
print("code:", CODE_DIR)

In [ ]:
# Ô K3 — tải ảnh (Zenodo) + CSV fold 0 (GitHub tác giả), kiểm tra MD5, giải nén ra đĩa tạm
import hashlib, zipfile, urllib.request
DATA_DIR   = os.environ.get("LAB_DATA", "/tmp/data")
LABELS_DIR = f"{DATA_DIR}/labels"
ZIP_LOCAL  = f"{DATA_DIR}/images.zip"
ZIP_URL    = "https://zenodo.org/records/7939060/files/images.zip?download=1"
ZIP_MD5    = "b7b30f96d466fba86016aa5a26606e0f"
os.makedirs(LABELS_DIR, exist_ok=True)

def md5sum(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

if not LOCAL_TEST and not os.path.exists(f"{DATA_DIR}/.images_ok"):
    if not os.path.exists(ZIP_LOCAL):
        print("Tải images.zip từ Zenodo...")
        urllib.request.urlretrieve(ZIP_URL, ZIP_LOCAL)
    got = md5sum(ZIP_LOCAL)
    assert got == ZIP_MD5, f"MD5 sai: {got}"
    print("MD5 đúng:", got)
    with zipfile.ZipFile(ZIP_LOCAL) as z:
        z.extractall(f"{DATA_DIR}/images")
    open(f"{DATA_DIR}/.images_ok", "w").close()

BASE_URL = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    if not os.path.exists(f"{LABELS_DIR}/{name}.csv"):
        urllib.request.urlretrieve(f"{BASE_URL}/{name}.csv", f"{LABELS_DIR}/{name}.csv")

IMAGES_DIR = max((r for r, _, fs in os.walk(f"{DATA_DIR}/images")), key=lambda r: len(os.listdir(r)))
print("IMAGES_DIR =", IMAGES_DIR, "|", sum(f.endswith(".jpg") for f in os.listdir(IMAGES_DIR)), "ảnh .jpg")
train_df, val_df, test_df = dataset.load_split(LABELS_DIR, fold=0)
dataset.check_split(train_df, val_df, test_df, IMAGES_DIR,
                    expected_total=None if LOCAL_TEST else dataset.TOTAL_IMAGES, verbose=False)
print("Kiểm tra chia dữ liệu: ĐẠT")

In [ ]:
# Ô K4 — cấu hình chung: GIỐNG HỆT Ô 0.4 + B2.x của lab_day2.ipynb (ConvNeXt-T, công thức nền T00)
BASE = dict(images_dir=IMAGES_DIR, labels_dir=LABELS_DIR, out_dir=RUNS_DIR, curves_dir=f"{OUT}/curves",
            pred_dir=f"{OUT}/predictions", logs_dir=f"{OUT}/logs", num_workers=2)
if LOCAL_TEST:   # chạy thử trên CPU: mạng không pretrained, ảnh nhỏ, 2 bước mỗi epoch
    BASE.update(expected_total=None, pretrained=False, img_size=64, batch_size=16, epochs=1,
                max_steps_per_epoch=2, num_workers=0)
T_BASE = dict(BASE, backbone="convnext_tiny")
print(Config(**T_BASE))

In [ ]:
# Ô K5 — Bước 2: mốc T00 (3 seed) + T01–T10 (mỗi cái chỉ khác T00 đúng một yếu tố, seed 0)
import traceback
RUNS = [
    ("T00", 0, "base",        {}),
    ("T00", 1, "base",        {}),
    ("T00", 2, "base",        {}),
    ("T01", 0, "frozen",      dict(init="frozen")),                                # A: đóng băng backbone, chỉ train head
    ("T02", 0, "scratch",     dict(init="scratch")),                               # A: từ đầu, không pretrained
    ("T03", 0, "trivial",     dict(aug="trivial")),                                # B: + TrivialAugmentWide
    ("T04", 0, "cutmix",      dict(mix="cutmix")),                                 # B: + CutMix (alpha 1.0)
    ("T05", 0, "dihedral",    dict(aug="dihedral")),                               # B: + lật dọc, xoay bội 90 độ
    ("T06", 0, "ls01",        dict(loss="ls", label_smoothing=0.1)),               # C: label smoothing 0,1
    ("T07", 0, "focal_g2",    dict(loss="focal", focal_gamma=2.0)),                # C: focal loss γ=2
    ("T08", 0, "cb_weighted", dict(loss="ce_weighted", class_weight_beta=0.9999)), # C: CE trọng số class-balanced
    ("T09", 0, "balanced",    dict(sampler="balanced")),                           # D: sampler cân bằng lớp
    ("T10", 0, "ema",         dict(ema_decay=0.999)),                              # F: EMA trọng số
]
failed = []
for exp_id, seed, desc, change in RUNS:
    try:
        run(Config(exp_id=exp_id, seed=seed, desc=desc, **dict(T_BASE, **change)))
    except Exception:
        traceback.print_exc()
        failed.append(f"{exp_id} seed{seed}")
print("Run lỗi:", failed or "không có")

In [ ]:
# Ô K6 — bảng tóm tắt Bước 2
t_df = train.collect_summaries(RUNS_DIR, "T")
t00 = t_df.loc[t_df.exp_id == "T00", "val_macro_f1"]
print(f"T00 ({len(t00)} seed): macro-F1 val = {t00.mean():.4f} ± {t00.std(ddof=1):.4f} (mean ± std)")
t_df["delta_vs_T00_mean"] = t_df["val_macro_f1"] - t00.mean()
cols = ["exp_id", "seed", "desc", "best_epoch", "val_macro_f1", "delta_vs_T00_mean", "val_top1",
        "val_balanced_acc", "val_ece", "f1_chinee", "f1_snake", "f1_negative", "train_time_per_epoch_s"]
t_df[cols].to_csv(f"{OUT}/logs/recipes_summary.csv", index=False)
print(t_df[cols].sort_values(["exp_id", "seed"]).round(4).to_string(index=False))

In [ ]:
# Ô K7 — gói kết quả nhỏ thành kaggle_buoc2.zip (KHÔNG có checkpoint .pt) để tải về và đưa lên Drive
import zipfile
from pathlib import Path
zp = Path(WORK) / "kaggle_buoc2.zip"
keep = {"done.json", "config.json", "history.csv", "val_logits.npy", "val_pred.csv"}
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for f in Path(RUNS_DIR).glob("T*/seed*/*"):
        if f.name in keep:
            z.write(f, Path("deepweeds_runs") / f.relative_to(RUNS_DIR))
    for sub in ("curves", "logs"):
        for f in Path(OUT, sub).rglob("*"):
            if f.is_file():
                z.write(f, Path("sub") / f.relative_to(OUT))
with zipfile.ZipFile(zp) as z:
    names = z.namelist()
print(zp, f"{zp.stat().st_size / 1e6:.1f} MB,", len(names), "file;",
      sum(n.startswith("deepweeds_runs/") and n.endswith("done.json") for n in names), "run đã xong")